# Mazure: seepage into a polder next to open water

The formula of Mazure follows from the steady state of Bruggeman 123.33 (see also the notebook 123.3x): first the head in the aquifer, then the seepage through the leaky layer, integrated over $x$ from 0 to infinity.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import timflow.steady as tfs
from scipy.integrate import quad

from bruggeman import Q_mazure, bruggeman_123_33, q_mazure

## Bruggeman 123.33: steady state after a sudden change of the surface water level

View the function.

In [ ]:
bruggeman_123_33

View the docstring to get a description of the input parameters

In [ ]:
help(bruggeman_123_33)

Define some aquifer parameters. The heads are relative to the polder level above the leaky layer, so `h` is the level of the open water above the polder level: $h = H_0 - H_{polder}$.

In [ ]:
k = 20.0  # hydraulic conductivity, m/d
D = 50.0  # thickness of aquifer, m
c = 500.0  # resistance of the leaky layer, d
h = 2.0  # level of the open water above the polder level, m

lab = np.sqrt(k * D * c)  # leakage factor, m
print(f"leakage factor: {lab:.0f} m")

Set up a steady `timflow` model: a leaky aquifer below the polder (polder level `hstar = 0`) with the open water at $x = 0$ as a `River1D` with level `h`. The aquifer extends to both sides, so the open water acts as a symmetry line; only $x \geq 0$ is used.

In [ ]:
def timflow_steady(c):
    ml = tfs.ModelXsection(naq=1)
    tfs.XsectionMaq(
        ml,
        x1=-np.inf,
        x2=np.inf,
        kaq=[k],
        c=[c],
        z=[1, 0, -D],
        topboundary="semi",
        hstar=0.0,
    )
    tfs.River1D(ml, xls=0.0, hls=h)
    ml.solve(silent=True)
    return ml


ml = timflow_steady(c)

Compare `timflow` to the analytical solution from Bruggeman, for a few resistances of the leaky layer.

In [ ]:
x = np.linspace(0, 3000, 101)
plt.figure(figsize=(10, 3))
for ci in [100.0, 500.0, 2000.0]:
    ha = bruggeman_123_33(x, h, k, D, ci)
    (ph,) = plt.plot(x, ha, label=f"c={ci:.0f} d")
    h_ml = timflow_steady(ci).headalongline(x[::10], 0.0)
    plt.plot(x[::10], h_ml[0], marker="x", ls="none", c=ph.get_color())
plt.plot([], [], marker="x", ls="none", color="k", label="timflow")
plt.legend(loc=(0, 1), frameon=False, ncol=6, fontsize="small")
plt.xlabel("x [m]")
plt.ylabel("head above polder level [m]")
plt.grid();

## The formula of Mazure

The seepage through the leaky layer into the polder is the head difference over the leaky layer divided by its resistance. With the head of 123.33 this gives the formula of Mazure:

$$q(x) = \frac{H(x) - H_{polder}}{c} = \frac{H_0 - H_{polder}}{c}\, e^{-x/\lambda}, \qquad \lambda = \sqrt{kDc}$$

with $q$ the seepage per unit area [m/d] (positive: upward, into the polder).

Source: Mazure, J.P. (1936). *Kwel en chloorbezwaar in de Wieringermeer*. Bijlage 10 in: *Geo-hydrologische gesteldheid van de Wieringermeer*. Rapporten en Mededeelingen betreffende de Zuiderzeewerken, No. 5: 67-131. Algemeene Landsdrukkerij, 's-Gravenhage. The equations are on p. 87-88 and in the appendix (Aanhangsel, § 1, p. 118), with $\lambda = \sqrt{kHc}$ and $\beta = \sqrt{kH/c}$; Mazure writes $p$ for the head in the aquifer, $h$ for the polder level and $H$ for the thickness of the aquifer. With open water at $x = 0$ the total seepage is the flow at $x = 0$: $s_0 = \beta\,(p_0 - h)$, the total seepage below.

Note: for seepage under a dike with a leaky layer on both sides (the bed of the open water and the polder), Mazure (§ 2, p. 119) gives $s_0 = \beta h / (2 + B/\lambda)$, with $B$ the width of the dike. This notebook is the case of open water in direct contact with the aquifer, as in Bruggeman 123.33.

In [ ]:
q_mazure

In [ ]:
help(q_mazure)

Compare to `timflow`: the seepage is the head in the aquifer minus the polder level (`hstar = 0`), divided by the resistance.

In [ ]:
q = q_mazure(x, h, k, D, c)
q_ml = ml.headalongline(x[::10], 0.0)[0] / c

plt.figure(figsize=(10, 3))
plt.plot(x, q * 1000, label="Mazure")
plt.plot(x[::10], q_ml * 1000, marker="x", ls="none", color="k", label="timflow")
plt.axvline(lab, color="grey", ls="--")
plt.text(lab, plt.ylim()[1], r" $\lambda$", va="top")
plt.legend(loc=(0, 1), frameon=False, ncol=6, fontsize="small")
plt.xlabel("x [m]")
plt.ylabel("seepage q [mm/d]")
plt.grid();

### Total seepage

Integrating the seepage over $x$ from 0 to infinity gives the total seepage into the polder per metre of open water:

$$Q = \frac{H_0 - H_{polder}}{c} \int_0^{\infty} e^{-x/\lambda}\, dx = \frac{H_0 - H_{polder}}{c}\, \lambda = (H_0 - H_{polder}) \sqrt{\frac{kD}{c}}$$

In [ ]:
Q_mazure

In [ ]:
help(Q_mazure)

Three ways to get the total seepage, which should be the same:

1. the formula of Mazure;
2. a numerical integration of $q(x)$ from 0 to infinity (`scipy.integrate.quad`);
3. `timflow`: all seepage into the polder comes from the open water, so it equals the horizontal discharge in the aquifer just next to the open water ($x = 0^+$).

In [ ]:
Q = Q_mazure(h, k, D, c)
Q_quad, _ = quad(q_mazure, 0, np.inf, args=(h, k, D, c))
Q_ml = ml.disvec(1e-6, 0.0)[0][0]  # Qx just next to the open water

print(f"Mazure:                {Q:.4f} m2/d")
print(f"numerical integration: {Q_quad:.4f} m2/d")
print(f"timflow, Qx at x = 0+: {Q_ml:.4f} m2/d")

The cumulative seepage from the open water up to distance $x$ is $Q\,(1 - e^{-x/\lambda})$: about 63% of the seepage enters the polder within one leakage factor from the open water, and 95% within three leakage factors.

In [ ]:
x_cum = np.linspace(0, 5 * lab, 201)
Q_cum = np.array([quad(q_mazure, 0, xi, args=(h, k, D, c))[0] for xi in x_cum])

plt.figure(figsize=(10, 3))
plt.plot(x_cum, Q_cum, label=r"$\int_0^x q\,dx$ (numerical)")
plt.plot(
    x_cum, Q * (1 - np.exp(-x_cum / lab)), "k:", label=r"$Q\,(1 - e^{-x/\lambda})$"
)
plt.axhline(Q, color="grey", label="Q (Mazure)")
for n in (1, 3):
    plt.axvline(n * lab, color="grey", ls="--")
    plt.text(n * lab, 0, f" {n}$\\lambda$", va="bottom")
plt.legend(loc=(0, 1), frameon=False, ncol=6, fontsize="small")
plt.xlabel("x [m]")
plt.ylabel("cumulative seepage [m$^2$/d]")
plt.grid();

The formula of Mazure, the numerical integration and `timflow` give the same total seepage. The total seepage grows with the transmissivity and decreases with the resistance of the leaky layer, as $\sqrt{kD/c}$: a less resistant leaky layer gives more seepage close to the open water, but the seepage zone becomes narrower.